# Qwen3.8-27B（原生全精度多模态 API 服务）

- **支持显卡**：A100 (40GB / 80GB)、H100 (80GB)、RTX 6000 (96GB)
- **默认配置**：[`Qwen/Qwen3.8-27B`](https://huggingface.co/Qwen/Qwen3.8-27B) BF16（40GB 显卡使用 FP8）、256K（`262144`）上下文、图像 / 视频理解、`reasoning_effort` 与工具调用
- **访问方式**：通过 Colab Secrets 自动读取 `CF_TUNNEL_TOKEN`，挂载 Cloudflare Tunnel 直连自有域名与 API
- **执行顺序**：`1. 环境检测` → `2. 安装 vLLM 推理引擎` → `3. 下载 Qwen3.8-27B 模型` → `4. 启动并挂载 API 服务`

| 接口路径 | 协议 | 客户端 |
| --- | --- | --- |
| `POST /v1/messages` | Anthropic Messages | Claude Code |
| `POST /v1/chat/completions` | OpenAI Chat Completions | OpenAI SDK、Cursor、Cline |
| `GET /v1/models` | OpenAI / Anthropic Models | 模型名 `Qwen/Qwen3.8-27B` 或 `qwen3.8-27b` |

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB 及以上显存（A100 40GB / 80GB、H100 80GB、RTX 6000 96GB）"

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 / H100 / RTX 6000 GPU。")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | 显存: {vram_gib:.1f} GiB | 系统内存: {ram_gib:.1f} GiB | 磁盘可用: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"当前显卡为 {gpu} ({vram_gib:.1f} GiB)，本笔记本需要 {REQUIREMENT}，请切换 runtime。")
if not bf16:
    raise RuntimeError("当前 GPU 不支持 BF16，请切换 runtime。")
if vram_gib < 70:
    print("[提示] 40GB 级显卡，第 3 步使用 FP8 模型。")
print(f"[OK] GPU 环境正常：{gpu} ({vram_gib:.1f} GiB)")

### 2. 安装 vLLM 推理引擎
- **功能**：安装 vLLM `0.32.0`、`hf_xet`、`openai` 与 `cloudflared` `2026.10.6`。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. 安装 vLLM 推理引擎 {display-mode: "form"}
import subprocess, sys, urllib.request
from pathlib import Path

VLLM_VERSION = "0.32.0"
CLOUDFLARED_VERSION = "2026.10.6"

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

run([sys.executable, "-m", "pip", "install", "-q", f"vllm=={VLLM_VERSION}", "huggingface_hub", "hf_xet", "openai"])

cloudflared_bin = Path("/usr/local/bin/cloudflared")
try:
    installed = subprocess.run([str(cloudflared_bin), "--version"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
except OSError:
    installed = ""
if CLOUDFLARED_VERSION not in installed.split():
    download = cloudflared_bin.with_name("cloudflared.download")
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        str(download),
    )
    download.chmod(0o755)
    download.replace(cloudflared_bin)

vllm_ver = subprocess.check_output([sys.executable, "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
print(f"vLLM: {vllm_ver} | cloudflared: {CLOUDFLARED_VERSION}")
print("[OK] vLLM 安装完成。")

### 3. 下载 Qwen3.8-27B 模型
- **功能**：按显存下载模型至 `/content/models`：80GB 及以上为 `Qwen/Qwen3.8-27B` BF16（约 55 GB），40GB 为 `Qwen/Qwen3.8-27B-FP8`（约 28 GB）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. 下载 Qwen3.8-27B 模型 {display-mode: "form"}
import os, shutil, time, torch
from pathlib import Path

os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import snapshot_download

MODEL_SPECS = {
    "BF16": {
        "repo_id": "Qwen/Qwen3.8-27B",
        "revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
        "local_dir": Path("/content/models/Qwen3.8-27B"),
        "min_free_gb": 58,
    },
    "FP8": {
        "repo_id": "Qwen/Qwen3.8-27B-FP8",
        "revision": "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a",
        "local_dir": Path("/content/models/Qwen3.8-27B-FP8"),
        "min_free_gb": 31,
    },
}

vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
precision = "BF16" if vram_gib >= 70 else "FP8"
spec = MODEL_SPECS[precision]
target_dir = spec["local_dir"]
target_dir.parent.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(target_dir.parent).free / 1024**3
if not (target_dir / "config.json").is_file() and free_gb < spec["min_free_gb"]:
    raise RuntimeError(f"磁盘空间不足：{spec['repo_id']} 需 {spec['min_free_gb']} GB，可用 {free_gb:.1f} GB。")

print(f"下载：{spec['repo_id']}（{precision}，显存 {vram_gib:.1f} GiB）")
started = time.time()
snapshot_download(
    repo_id=spec["repo_id"],
    revision=spec["revision"],
    local_dir=str(target_dir),
)
SELECTED_QWEN38_DIR = str(target_dir)
print(f"[OK] 模型准备完成：{target_dir}（用时 {time.time() - started:.0f} 秒）")

### 4. 启动并挂载 API 服务
- **功能**：后台启动 `vllm serve`（`127.0.0.1:8188`，`262144` 上下文，MTP 投机采样），读取 `CF_TUNNEL_TOKEN` 启动 Cloudflare Tunnel 并检查公网访问，提供 OpenAI（`/v1/chat/completions`）与 Anthropic（`/v1/messages`）接口。
- **要求**：在 Colab 左侧 **🔑 Secrets** 中添加 `CF_TUNNEL_TOKEN` 并开启访问权限。
- **使用**：直接运行，输出 Claude Code 与 OpenAI 调用配置。

In [ ]:
# @title 4. 启动并挂载 API 服务 {display-mode: "form"}
import json, os, re, subprocess, time, urllib.request
from pathlib import Path

PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
LOG = Path("/content/vllm.log")
LAUNCH_CONFIG = Path("/content/vllm.launch.json")
PROCESS_PATTERN = "vllm serve|vllm.entrypoints"

CF_BIN = Path("/usr/local/bin/cloudflared")
CF_LOG = Path("/content/cloudflared.log")

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token):
    if not CF_BIN.exists():
        raise RuntimeError("未找到 cloudflared，请先运行第 2 步。")
    stop_processes("cloudflared tunnel", 10)
    rotate_log(CF_LOG)
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            [
                str(CF_BIN),
                "tunnel",
                "--no-autoupdate",
                "--protocol", "http2",
                "--edge-ip-version", "4",
                "run",
                "--url", BASE,
                "--token", token,
            ],
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    deadline = time.time() + 30
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(tail(CF_LOG, 4000))
            raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
        log_text = read_text(CF_LOG)
        if "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = read_text(CF_LOG)
            configs = [line for line in log_text.splitlines() if "Updated to new configuration" in line]
            config = configs[-1].replace('\\"', '"') if configs else ""
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"(?:(?!"hostname").)*?"service"\s*:\s*"([^"]+)"', config)
            for host, service in routes:
                print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): https://{host} -> {service}")
            hosts = [host for host, service in routes if f":{PORT}" in service]
            if not routes:
                print("[OK] Cloudflare Tunnel 已连接 (HTTP/2)，请通过你在 Cloudflare 绑定的域名访问。")
            elif not hosts:
                print(f"[WARN] 没有域名转发到本服务的 {PORT} 端口！")
                print(f"       请前往 Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname，将 Service URL 改为 {BASE}")
            return f"https://{hosts[0]}" if hosts else "https://<你的CF域名>"
        time.sleep(1)
    print(f"[WARN] Cloudflare Tunnel 30 秒内未确认连接，仍在后台运行，详见 {CF_LOG}。")
    return "https://<你的CF域名>"

def check_public_url(url, route):
    if "<" in url:
        return
    error = None
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            request = urllib.request.Request(url + route, headers={"User-Agent": "colab-tunnel-check"})
            with urllib.request.urlopen(request, timeout=10):
                print(f"[OK] 公网访问正常：{url}")
                return
        except Exception as e:
            error = e
        time.sleep(3)
    print(f"[WARN] 公网地址 60 秒内无法访问：{url}{route}（{error}）")

cf_token = resolve_cf_token()
if not cf_token:
    raise RuntimeError("未检测到 CF_TUNNEL_TOKEN，请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限。")

model_dir = globals().get("SELECTED_QWEN38_DIR") or "/content/models/Qwen3.8-27B"
if not (Path(model_dir) / "config.json").is_file():
    raise RuntimeError(f"未找到模型目录 {model_dir}，请先运行第 3 步下载模型。")

def ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

cmd = [
    "vllm", "serve", model_dir,
    "--served-model-name", "Qwen/Qwen3.8-27B", "qwen3.8-27b",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--max-model-len", "262144",
    "--gpu-memory-utilization", "0.93",
    "--enable-prefix-caching",
    "--reasoning-parser", "qwen3",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--trust-remote-code",
    "--speculative-config", '{"method":"mtp","num_speculative_tokens":3}',
]

launch_config = json.dumps(cmd)
running = ready() or (process_running(PROCESS_PATTERN) and ready(timeout=30))
if running and read_text(LAUNCH_CONFIG) != launch_config:
    print("启动参数已变化，正在按新参数重启 vLLM 服务...")
    stop_processes(PROCESS_PATTERN, 120)
    running = False

if running:
    print("[OK] vLLM 服务已经在运行。")
else:
    stop_processes(PROCESS_PATTERN, 120)
    rotate_log(LOG)
    with LOG.open("w") as log:
        process = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    LAUNCH_CONFIG.write_text(launch_config)
    print("正在加载 27B 权重并初始化 KV Cache（约需 1~3 分钟）...")
    deadline = time.time() + 900
    while time.time() < deadline:
        if process.poll() is not None:
            print(tail(LOG))
            raise RuntimeError(f"vLLM 启动失败（退出码 {process.returncode}），请查看上方日志。")
        if ready():
            print("[OK] vLLM 服务启动成功。")
            break
        time.sleep(3)
    else:
        print(tail(LOG))
        raise TimeoutError("vLLM 15 分钟内未就绪，请查看上方日志。")

host_url = start_cf_tunnel(cf_token)
check_public_url(host_url, "/health")

print(f"""
=== 1. Claude Code (cc) 直连配置（Anthropic /v1/messages 原生协议）===
export ANTHROPIC_BASE_URL="{host_url}"
export ANTHROPIC_AUTH_TOKEN="EMPTY"
export ANTHROPIC_API_KEY="EMPTY"
export ANTHROPIC_MODEL="qwen3.8-27b"
export ANTHROPIC_SMALL_FAST_MODEL="qwen3.8-27b"
export CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC=1
claude --model qwen3.8-27b

=== 2. OpenAI 兼容接口配置（/v1/chat/completions）===
Base URL : {host_url}/v1
Model    : qwen3.8-27b （或 Qwen/Qwen3.8-27B）
API Key  : EMPTY
""")